In [4]:
import os
import numpy as np
import warnings 

from datetime import datetime

from ObservationDataUtil import ( 
     ObsDataReader,
    
)
from ModelDataUtil import (
    ModelDataReader,
    EnsembleMetricEvaluator
)

In [5]:
if __name__ == "__main__":
    #top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    #out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    #fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"

    top_path = "/pscratch/sd/z/zhan391/e3sm_dart"
    out_path = "/pscratch/sd/z/zhan391/e3sm_dart/diag_dart"
    fig_path = "/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"
    
    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    regnam = "global"
    plot_type = "model_vs_obs"
    period = "201112-201112"  # monthly format
    frequency = "monthly"
    
    derive_monthly = True 

    
    # === Variable dictionary ===
    var_dict = {
        'H2OSOI': {'alias' :'H2OSOI',  'unit':'mm',             'level': np.linspace(-10, 10, 51), 'comp': 'lnd', 'ref': "ESA_CCI"},
        'SOILWATER_10CM':{'alias':'SOILWATER_10CM','unit':'mm', 'level': np.linspace(-10, 10, 51), 'comp': 'lnd', 'ref': "CPC_SOM"},
        'TSA':    {'alias' : 'T2M',    'unit': '$^o$C',         'level': np.linspace(-10, 10, 51), 'comp': 'lnd', 'ref': "MODIS_LST"},
    }
    
    model_labels = {
        'CTRLEN10': "CTRL (ENS=10)",
        'DARTEN10': "EAM-DART (ENS=10)",
        'DARTEN20': "EAM-DART (ENS=20)",
        'DARTEN40': "EAM-DART (ENS=40)",
        'DARTEN80': "EAM-DART (ENS=80)",
    }

    model_list = ['CTRLEN10', 'DARTEN20', 'DARTEN40']

    # Ensure output path exists
    os.makedirs(out_path, exist_ok=True)

    # Optional: save run configuration for reproducibility
    config_file = os.path.join(out_path, "diagnostic_run_config.txt")
    with open(config_file, "w") as f:
        f.write(f"Run Time: {datetime.now().isoformat()}\n")
        f.write(f"Variables: {list(var_dict.keys())}\n")
        f.write(f"Models: {model_list}\n")
        f.write(f"Region: {regnam}, Period: {period}, Frequency: {frequency}\n")
        f.write(f"Experiment Base: {exp_base}\n")

    # === Loop over variables ===
    for var, meta in var_dict.items():
        try:
            print(f"[{datetime.now().isoformat()}] Processing {var} using {meta['ref']}")

            component = meta.get('comp', 'atm')
            ref_var = meta.get('alias', var)
            ref_dataset = meta.get('ref', 'ERA5')

            # Load reference data
            try:
                obs_reader =  ObsDataReader(
                    obsname=ref_dataset,
                    period=period,
                    frequency=frequency
                )
                obs = obs_reader.read(var=ref_var)
            except FileNotFoundError:
                print(f"[WARN] Skipping {var} — observation not found for {ref_var} ({ref_dataset})")
                continue
                
            # Load model ensemble data
            model_reader = ModelDataReader(
                base_path=top_path,
                exp_base=exp_base,
                regnam=regnam,
                frequency=frequency,
                period=period,
                component=component,
                derive_monthly=derive_monthly
            )
            data_dict = model_reader.read_variable_across_experiments(var, model_list)
            #print(data_dict)
            #print(obs.min(),obs.max())
            #print(xxxx)
            
            # Evaluate metrics and save
            evaluator = EnsembleMetricEvaluator(
                obs=obs,
                model_dict=data_dict,
                component=component,
                ref_dataset=ref_dataset,
                var=var,
                vunit=meta['unit'],
                output_path=out_path
            )

        except Exception as e:
            print(f"[ERROR] Failed to process {var}: {e}")


[2025-06-10T14:16:57.023711] Processing H2OSOI using ESA_CCI
Parsed time range: 2011-12-01 00:00:00 to 2011-12-01 23:59:59
2011
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
Parsed time range: 2011-12-01 00:00:00 to 2011-12-01 23:59:59
read CTRLEN10 data from /pscratch/sd/z/zhan391/e3sm_dart/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/lnd/180x360_aave/ts/daily
read CTRLEN10 data from /pscratch/sd/z/zhan391/e3sm_dart/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/lnd/180x360_aave/ts/6hourly
[ERROR] Failed to process H2OSOI: No files found for EN01: tried daily and 6hourly.
[2025-06-10T14:16:57.116010] Processing SOILWATER_10CM using CPC_SOM
Parsed time range: 2011-12-01 00:00:00 to 2011-12-01 23:59:59
2011
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
Parsed time range: 2011-12-01 00:00:00 to 2011-12-01 23:59:59
read CTRLEN10 data from /pscratch/sd/z/zhan391/e3sm_dart/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/a